In [ ]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

sys.path.append(os.path.abspath("../dataset and other libs"))
from cleaningcls import clean_cls
from ctf import CorrelationThresholdFilter
from pridict_thresh import pridict_thresh


In [ ]:
# Load raw dataset
df = pd.read_csv('D:/Repos/Chrun-Pridictor/dataset and other libs/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Separate features (X) and target (y)
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print("\nTarget distribution in train set:")
print(y_train.value_counts(normalize=True).round(3))

In [ ]:

pipeline_dt = Pipeline([
    ('clean', clean_cls()),
    ('ctf', CorrelationThresholdFilter(threshold=0.14)),
    ('pridict_thresh', pridict_thresh(RandomForestClassifier(
            n_estimators=200,           # More trees for stability
            max_depth=8,             # Prevent hyper-specific splits
            class_weight='balanced',    # Handle churn class imbalance
            random_state=42), thresh=0.64))])    
                             # Your custom high-precision threshold
pipeline_dt.fit(X_train, y_train)
y_prob_dt = pipeline_dt.predict_proba(X_test)[:,1]
y_pred_dt = pipeline_dt.predict(X_test)
# y_pred_dt = (pipeline_dt.predict_proba(X_test)[: , 1] > 0.45).astype(int)

auc_dt = roc_auc_score(y_test, y_prob_dt)
print('=== Pipeline Evaluation ===')
print(f'ROC-AUC Score: {auc_dt:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_dt))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt))

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.arange(0.25, 0.66, 0.01)
for t in thresholds:
    y_pred = (y_prob_dt > t).astype(int)
    print(f"thresh={t:.2f} | P={precision_score(y_test,y_pred):.2f} | R={recall_score(y_test,y_pred):.2f} | F1={f1_score(y_test,y_pred):.2f}")


In [ ]:
import pandas as pd

# 1. Get the trained model from your pipeline
model = pipeline_dt['pridict_thresh'].model

# 2. Get the feature importances (scores of how much the tree uses each feature)
importances = model.feature_importances_

# 3. Get the feature names from your pipeline's preprocessing steps

feature_names = pipeline_dt['ctf'].get_feature_names_out()


# 4. Combine them into a clean DataFrame and sort by most important
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

active_features = feature_importance_df

print("=== Features Used by the Decision Tree (Ranked by Importance) ===")
print(active_features.to_string(index=False))

In [ ]:
feature_names

In [ ]:

sample_output = pipeline_dt['clean'].transform(X_train.head(5))
sample_output = pipeline_dt['ctf'].transform(sample_output)
feature_names = list(sample_output.columns)

In [74]:
from trace_path import trace_customer_path
import trace_path

X_transformed = pipeline_dt['clean'].transform(X_train.head(1))
X_filtered = pipeline_dt['ctf'].transform(X_transformed)
trace_customer_path(pipeline_dt, X_filtered )

ℹ️  RandomForest detected — tracing through tree #0 of 200
Customer's Decision Path (Node IDs): [  0 134 135 136 137 138 139 140 142]
Final Leaf Node: 142
Node 0 (Split Node):
  - Feature checked: 'TotalCharges'
  - Tree rule: <= 232.0750
  - Customer's value: 1701.65  →  Goes RIGHT (>)
----------------------------------------
Node 134 (Split Node):
  - Feature checked: 'Contract_Two year'
  - Tree rule: <= 0.5000
  - Customer's value: 0.0  →  Goes LEFT (<=)
----------------------------------------
Node 135 (Split Node):
  - Feature checked: 'MonthlyCharges'
  - Tree rule: <= 69.9000
  - Customer's value: 49.2  →  Goes LEFT (<=)
----------------------------------------
Node 136 (Split Node):
  - Feature checked: 'InternetService_No'
  - Tree rule: <= 0.5000
  - Customer's value: 0.0  →  Goes LEFT (<=)
----------------------------------------
Node 137 (Split Node):
  - Feature checked: 'InternetService_Fiber optic'
  - Tree rule: <= 0.5000
  - Customer's value: 0.0  →  Goes LEFT (<=)
--

c:\Users\HomePC\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\HomePC\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
